In [3]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

import pandas as pd
import numpy as np

images = pd.read_csv('../../data/images.csv', on_bad_lines='skip', nrows=1000).fillna('')
images['id'] = images.filename.str.extract(r'(\d+)').astype(int)

styles = pd.read_csv('../../data/styles.csv', on_bad_lines='skip', nrows=1000).fillna('')
styles['context'] = styles.productDisplayName
styles['context'] += ' #' + styles.gender
styles['context'] += ' #' + styles.masterCategory
styles['context'] += ' #' + styles.subCategory
styles['context'] += ' #' + styles.articleType
styles['context'] += ' #' + styles.baseColour
styles['context'] += ' #' + styles.season
styles['context'] += ' #' + styles.usage

In [4]:
X_train, y_train = styles.context.values, images.link.values
X_test, y_test = [], []

testing = list(zip(X_train, y_train))
np.random.shuffle(testing)

for X, y in testing:
    original = X.split(' ')
    terms = original.copy()
    np.random.shuffle(terms)
    keepers = terms[:-5]
    keep = ' '.join([text for text in original if text in keepers and '#' not in text])
    X_test.append(keep)
    y_test.append(y)
    if len(X_test) >= len(X_train) * 0.2:
        break

X_test = np.array(X_test)
y_test = np.array(y_test)

In [8]:
import sys, os
path_to_root = os.path.abspath("../../")

if path_to_root not in sys.path:
    sys.path.append(path_to_root)

from models.recommender import Recommender

model = Pipeline(
  steps=[
    ('vectorizer', TfidfVectorizer(stop_words='english')),
    ('recommender', Recommender(n_neighbors=10))
  ]
)

model.fit(X_train, y_train)
model.score(X_test, y_test, k=5)


np.float64(0.8742154099026233)

In [9]:
model.predict(['Lakme Nine to Makeup Shell Foundation and'])

array([['http://assets.myntassets.com/assets/images/55233/2018/5/3/11525323433330-Lakme-Nine-to-Five-Flawless-Makeup-Shell-Foundation-7251525323433284-1.jpg',
        'http://assets.myntassets.com/assets/images/55234/2018/5/3/11525323471619-Lakme-9to5-Flawless-Makeup-Marble-Foundation-7751525323471576-1.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/853a100f5d3d8c1e8ef88b41ca0d2c88_images.jpg',
        'http://assets.myntassets.com/assets/images/55833/2019/1/10/b111ce8e-7f66-4319-b753-90a5580f08221547122964116-Colorbar-Glamour-Radiant-Glow-001-6691547122964016-1.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/Lakme-Absolute-Matte-Merlot-Lipstick-45_bc13569d8288f127c70c917c9c75bb24_images.jpg',
        'http://assets.myntassets.com/assets/images/57793/2018/4/27/11524809214196-Lotus-Herbals-Natural-Blend-Botanical-Compact-550-2491524809214143-1.jpg',
        'http://assets.myntassets.com/v1/images/style/properties/Lakme-Absolute-Matte-Milan-